In [11]:
import os
from pathlib import Path
import mysql.connector
import numpy as np
from dotenv import load_dotenv
import pandas as pd
from IPython.display import display
from sqlalchemy import URL, create_engine, text

In [12]:
def trouver_racine_projet():
    dossier_actuel = Path.cwd()

    for dossier in [dossier_actuel, *dossier_actuel.parents]:
        if (dossier / ".env").exists():
            return dossier

    raise FileNotFoundError(
        "Impossible de trouver le fichier .env"
    )


PROJECT_ROOT = trouver_racine_projet()
load_dotenv(PROJECT_ROOT / ".env")

print("Projet :", PROJECT_ROOT)

Projet : /Users/seresiaka/Downloads/ml_immobilier


In [13]:
load_dotenv(PROJECT_ROOT / ".env")

True

In [14]:
url_mysql = URL.create(
    drivername="mysql+mysqlconnector",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(
    url_mysql,
    pool_pre_ping=True,
)

In [15]:
requete = text("""
    SELECT
        l.id_logement,
        q.nom AS quartier,
        l.surface_m2,
        l.nombre_pieces,
        l.nombre_chambres,
        l.age_batiment,
        l.distance_centre_km,
        l.garage,
        l.jardin_m2,
        q.indice_attractivite,
        l.prix_eur
    FROM logements AS l
    JOIN quartiers AS q
        ON q.id_quartier = l.id_quartier
    ORDER BY l.id_logement
""")

In [16]:
with engine.connect() as connexion:
    df = pd.read_sql_query(
        sql=requete,
        con=connexion,
    )

engine.dispose()

In [17]:
df.head()

,id_logement,quartier,surface_m2,nombre_pieces,nombre_chambres,age_batiment,distance_centre_km,garage,jardin_m2,indice_attractivite,prix_eur
0,1,Centre,168.66,8,4,54,4.49,1,0.00,0.95,685773.33
1,2,Périphérie,197.85,7,3,89,27.78,1,0.00,0.50,624076.08
2,3,Périphérie,204.61,8,4,45,14.68,1,288.97,0.50,894014.95
3,4,Universitaire,123.21,5,2,22,8.16,0,289.08,0.65,642413.82
4,5,Universitaire,126.45,6,3,13,7.62,1,120.76,0.65,665509.25
